In [ ]:
# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import shap
from lime.lime_tabular import LimeTabularExplainer
from anchor import anchor_tabular
import openai
import matplotlib.pyplot as plt

# STEP 2: LOAD AND PREPARE THE CARDIO DATASET

# Load dataset
df = pd.read_csv("cardio_train.csv", sep=';')

# Convert age from days → years
df['age'] = (df['age'] / 365).round().astype(int)

print("Dataset shape:", df.shape)
print(df.head())

# Separate features (X) and target (y)
X = df.drop(columns=['cardio', 'id'])
y = df['cardio']

In [ ]:
# BASIC MISSING VALUE CHECKS
print("\nTotal missing values in the dataset:")
print(df.isnull().sum().sum())

print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# CHECK FOR CLASS IMBALANCE

# Count each class
class_counts = y.value_counts()

# Display counts and proportions
print("\nClass distribution:")
print(class_counts)
print("\nClass proportions (%):")
print(round((class_counts / len(y)) * 100, 2))

# Visualize class balance
plt.figure(figsize=(5,4))
plt.bar(class_counts.index.astype(str), class_counts.values, color=['#4c72b0', '#dd8452'])
plt.title("Distribution of Target Variable (cardio)")
plt.xlabel("Class (0 = No CVD, 1 = CVD)")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
df.describe()

In [ ]:
# Drop ID column if it exists
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

# Define features and label
X = df.drop('cardio', axis=1)
y = df['cardio']

# Optional: scale numerical features for LIME/Anchors stability
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#STEP 3: TRAIN XGBOOST MODEL

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# STEP 4A: SHAP EXPLANATIONS

import shap
import xgboost as xgb

# Get Booster from trained model
booster = model.get_booster()

# Define a wrapper for booster predictions
def booster_predict(data_as_df):
    # Ensure feature names are a list of strings
    feature_list = list(data_as_df.columns)
    dmatrix = xgb.DMatrix(data_as_df, feature_names=feature_list)
    return booster.predict(dmatrix)

# Create SHAP Explainer
explainer_shap = shap.Explainer(booster_predict, X_train)

# Compute SHAP values for test data
shap_values = explainer_shap(X_test)

# Global summary plot
shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns)

In [ ]:
#SHAP summary values
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean_Abs_SHAP': mean_abs_shap
}).sort_values('Mean_Abs_SHAP', ascending=False)

print(importance_df.head(10))

In [ ]:
shap.plots.force(shap_values[1:100])

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
for feature in ["age", "ap_hi", "cholesterol", "gluc", "weight"]:
    shap.dependence_plot(feature, shap_values.values, X_test)

In [ ]:
# Patient index
i = 12000
patient_data = X.iloc[X_test.index[i]]
print(patient_data)

In [ ]:
# Force plot

#Model prediction
patient_pred_prob = model.predict_proba(X_test.iloc[i:i+1])[0,1]
patient_pred_label = "Disease (CVD)" if patient_pred_prob >= 0.5 else "No Disease"

print(f"Patient {i} → Predicted Probability: {patient_pred_prob*100:.2f}%")
print(f"Predicted Class: {patient_pred_label}")

# Enable JS for interactive plots
shap.initjs()

# Force plot for one patient
shap.plots.force(shap_values[i])


In [ ]:
import matplotlib.pyplot as plt
import shap

# Create figure FIRST
plt.figure(figsize=(10, 3))

# Draw SHAP force plot (static)
shap.force_plot(
    shap_values.base_values[i],
    shap_values.values[i],
    X_test.iloc[i],
    matplotlib=True
)

# Save figure
plt.savefig("shap_local.png", dpi=300, bbox_inches="tight", facecolor="white")

# Close
plt.close()


In [ ]:
# Waterfall plot
shap.plots.waterfall(shap_values[i], max_display=15)

In [ ]:
import matplotlib.pyplot as plt
import shap

# Create the waterfall plot
shap.plots.waterfall(shap_values[i], max_display=15)

# Save as PNG
plt.savefig("patient_12000_waterfall.png", dpi=300, bbox_inches="tight")
plt.close()


In [ ]:
# TABLE VERSION OF SHAP VALUES
shap_df = pd.DataFrame({
    "Feature": X_test.columns,
    "SHAP_value": shap_values.values[i]
}).sort_values("SHAP_value", ascending=False)

shap_text = shap_df.to_string(index=False)
print(shap_text)

In [ ]:
shap.plots.decision(
    shap_values[i].base_values,
    shap_values[i].values
)


In [ ]:
# STEP 4B: LIME EXPLANATIONS

patient_instance = X_test.iloc[i]

# Display features
print("Explaining patient", i)
print(patient_instance)
print("\nOriginal (unscaled) data:\n", X.iloc[X_test.index[i]])
print("True label:", y_test.iloc[i])

explainer_lime = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=["No Disease", "Disease"],
    mode="classification"
)

exp_lime = explainer_lime.explain_instance(
    data_row=patient_instance.values,
    predict_fn= model.predict_proba   # make sure this is your XGBoost model
)

exp_lime.show_in_notebook(show_table=True)
lime_text = str(exp_lime.as_list())
print("\nLIME explanation:", lime_text)

In [ ]:
# ============================
# STEP 4B: LIME EXPLANATIONS
# ============================

# Select patient index
i = i  # keep your existing index logic

# RAW (unscaled) patient data
patient_raw = X.iloc[X_test.index[i]]

print("Explaining patient", i)
print(patient_raw)
print("True label:", y_test.iloc[i])

# LIME explainer MUST see raw data
explainer_lime = LimeTabularExplainer(
    training_data=X.values,                 # RAW training data
    feature_names=X.columns.tolist(),
    class_names=["No Disease", "Disease"],
    mode="classification",
    discretize_continuous=True              # forces real value bins
)

# Explain instance (raw → scaled inside predict_fn)
exp_lime = explainer_lime.explain_instance(
    data_row=patient_raw.values,
    predict_fn=lambda x: model.predict_proba(scaler.transform(x)),
    num_features=10
)

# Show explanation
exp_lime.show_in_notebook(show_table=True)

# Save plot with real values
fig = exp_lime.as_pyplot_figure()
plt.title("Local explanation for class Disease")
plt.tight_layout()
plt.savefig("LimeExplain.png", dpi=300, bbox_inches="tight")
plt.show()

# Text explanation
lime_text = exp_lime.as_list()
print("\nLIME explanation:", lime_text)


In [ ]:
import pandas as pd
from IPython.display import display

lime_list = exp_lime.as_list()   # list of (feature, contribution)

lime_df = pd.DataFrame(lime_list, columns=["Feature (condition)", "Contribution"])
display(lime_df)

In [ ]:
# STEP 4C: ANCHORS EXPLANATIONS

from anchor import anchor_tabular

#Prepare instance to explain
instance = X_test.iloc[i].values.reshape(1, -1)

print(f"\nExplaining patient index {i}")
print(X_test.iloc[i])
print("True Label:", y_test.iloc[i])

#Create explainer
explainer_anchor = anchor_tabular.AnchorTabularExplainer(
    class_names=["No Disease", "Disease"],
    feature_names=X_train.columns.tolist(),
    train_data=X_train.values
)

#Explain the selected patient
exp_anchor = explainer_anchor.explain_instance(
    X_test.iloc[i].values,
    model.predict,
    threshold=0.95     
)

# 5. Show output

anchor_text = "\n".join([
    f"Anchor Rule: {exp_anchor.names()}",
    f"Precision:   {exp_anchor.precision():.3f}",
    f"Coverage:    {exp_anchor.coverage():.3f}"
])

print("\nGenerated Anchor Text:\n", anchor_text)

In [ ]:
# ==========================================
# STEP 4C: ANCHOR EXPLANATIONS (FINAL VERSION)
# ==========================================

import pandas as pd
from anchor import anchor_tabular

# ------------------------------------------
# 1. RAW (unscaled) data
# ------------------------------------------
X_raw = X.copy()   # original dataframe (NO scaling)

# Patient index
i = 12000   # or keep your own index logic

instance_raw = X_raw.iloc[X_test.index[i]].values.reshape(1, -1)

print(f"\nExplaining patient index {i}")
print(X_raw.iloc[X_test.index[i]])
print("True Label:", y_test.iloc[i])

# ------------------------------------------
# 2. Prediction wrapper (fixes warnings)
# ------------------------------------------
def predict_anchor(x):
    # restore column names (removes sklearn warnings)
    x_df = pd.DataFrame(x, columns=X.columns)
    x_scaled = scaler.transform(x_df)
    return model.predict(x_scaled)

# ------------------------------------------
# 3. Create Anchor explainer (RAW data)
# ------------------------------------------
explainer_anchor = anchor_tabular.AnchorTabularExplainer(
    class_names=["No Disease", "Disease"],
    feature_names=X_raw.columns.tolist(),
    train_data=X_raw.values
)

# ------------------------------------------
# 4. Explain instance
# ------------------------------------------
exp_anchor = explainer_anchor.explain_instance(
    instance_raw[0],
    predict_anchor,
    threshold=0.95
)

# ------------------------------------------
# 5. Output explanation (REAL values)
# ------------------------------------------
anchor_text = "\n".join([
    f"Anchor Rule: {exp_anchor.names()}",
    f"Precision:   {exp_anchor.precision():.3f}",
    f"Coverage:    {exp_anchor.coverage():.3f}"
])

print("\nGenerated Anchor Explanation:\n", anchor_text)


In [ ]:
import matplotlib.pyplot as plt
import shap

# Anchors summary plot
plt.figure()
plt.text(0.1, 0.6, "\n".join(exp_anchor.names()), fontsize=12)
plt.axis("off")

In [ ]:
from IPython.display import HTML, display
display(HTML(exp_anchor.as_html()))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No CVD", "CVD"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - XGBoost Model")
plt.show()

# Print classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No CVD", "CVD"]))


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Create figure
fig, ax = plt.subplots(figsize=(6, 5))

# Display confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No CVD", "CVD"]
)
disp.plot(cmap="Blues", ax=ax, colorbar=True)

# Title
ax.set_title("Confusion Matrix – XGBoost Model")

# Save as PNG (HIGH QUALITY)
plt.savefig(
    "confusion_matrix_xgboost.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

# Show (optional)
plt.show()

# Close figure
plt.close()

# Print classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No CVD", "CVD"]))


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Get model probabilities for positive class (CVD)
y_proba = model.predict_proba(X_test)[:, 1]

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

# Create figure
plt.figure(figsize=(5, 4))

# Plot ROC curve
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc_score:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", label="Random Guessing")

# Labels and title
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
#plt.title("ROC Curve – XGBoost Model")

# Legend and grid
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# Save as PNG
plt.savefig(
    "roc_curve_xgboost_kaggle.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

# Show (optional)
plt.show()

# Close figure
plt.close()

print(f"Area Under the Curve (AUC): {auc_score:.3f}")


In [ ]:
plt.figure(figsize=(2.0567, 1.37))  # width x height in inches

# Plot ROC curve
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc_score:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", label="Random Guessing")

# Labels and title
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

# Legend and grid
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# Save as PNG with 300 dpi
plt.savefig(
    "roc_curve_xgboost_kaggle.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close()


In [ ]:
#STEP 5: LLM-BASED REPORT GENERATION (OpenAI API)

#from openai import OpenAI
#client = OpenAI(api_key="")

import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


def generate_report(xai_output, audience="doctor"):
    if audience == "doctor":
        prompt = f"""
        Provide a brief clinical interpretation of these XAI results for a doctor (not more than 100 words):
        {xai_output}
        """
    else:
        prompt = f"""
        Briefly explain these XAI results in clear, friendly language for a patient (not more than 100 words):
        {xai_output}
        """

    response = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
#Step 6: Generate Results

# SHAP explanation reports only
method = "SHAP"
result_text = shap_text

print("="*80)
print(f"\n {method} EXPLANATION REPORTS\n")
print("="*80)

# Generate doctor and patient reports separately
doc_report = generate_report(result_text, "doctor")
patient_report = generate_report(result_text, "patient")

print(f"\nDoctor Report ({method}):\n{doc_report}\n")
print(f"\nPatient Report ({method}):\n{patient_report}\n")

In [ ]:
# LIME explanation reports only
method = "LIME"
result_text = lime_text

# Convert patient features into text for reporting
patient_info = patient_instance.to_frame(name="Value")
patient_info_str = patient_info.to_string()

# 3️Build a unified text block containing:
full_text_for_report = (
    "PATIENT FEATURE VALUES:\n"
    + patient_info_str
    + "\n\nLIME EXPLANATION:\n"
    + result_text
)
print("="*80)
print(f"\n {method} EXPLANATION REPORTS\n")
print("="*80)

# Generate doctor and patient reports separately
doc_report = generate_report(result_text, "doctor")
patient_report = generate_report(result_text, "patient")

print(f"\nDoctor Report ({method}):\n{doc_report}\n")
print(f"\nPatient Report ({method}):\n{patient_report}\n")

In [ ]:
# Anchors explanation reports only
method = "Anchors"
result_text = anchor_text

print("="*80)
print(f"\n {method} EXPLANATION REPORTS\n")
print("="*80)

# Generate doctor and patient reports separately
doc_report = generate_report(result_text, "doctor")
patient_report = generate_report(result_text, "patient")

print(f"\nDoctor Report ({method}):\n{doc_report}\n")
print(f"\nPatient Report ({method}):\n{patient_report}\n")

In [ ]:
#STEP 7: COMPARISON TABLE

comparison = pd.DataFrame({
    "Method": ["SHAP", "LIME", "Anchors"],
    "Output_Format": ["Feature contribution values", "Local linear approximation", "IF–THEN rules"],
    "Interpretability": ["High (quantitative)", "Medium–High (visual)", "High (logical rules)"],
    "Best_For": ["Global + local explanations", "Local single predictions", "Simple rule-based explanations"]
})
print("\n Comparison of XAI Methods:\n")
display(comparison)

In [ ]:
from docx import Document
from docx.shared import Inches

for method, result_text in methods.items():
    doc = Document()
    doc.add_heading(f'{method} Explainable AI Report', level=1)

    # Doctor section
    doc.add_heading('Doctor-Oriented Report', level=2)
    doc_report_text = generate_report(result_text, "doctor")
    doc.add_paragraph(doc_report_text)

    # Patient section
    doc.add_heading('Patient-Oriented Report', level=2)
    patient_report_text = generate_report(result_text, "patient")
    doc.add_paragraph(patient_report_text)

    # Add corresponding figure
    if method == "SHAP":
        doc.add_picture("shap_summary.png", width=Inches(5.5))
    elif method == "LIME":
        doc.add_picture("lime_instance.png", width=Inches(5.5))
    elif method == "Anchors":
        doc.add_picture("anchors_rules.png", width=Inches(5.5))

    output_path = f"{method}_XAI_Report.docx"
    doc.save(output_path)
    print(f"Saved {output_path}")